<a href="https://colab.research.google.com/github/syedahijabzahra/DevelopersHub-AI-ML-Internship/blob/main/Mental_Health_Chatbot/Mental_Health_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ============================================
# MENTAL HEALTH SUPPORT CHATBOT - FINAL WORKING VERSION
# Compatible with latest Transformers library
# ============================================

# Step 1: Install dependencies
!pip install transformers datasets accelerate torch pandas -q

import pandas as pd
import torch
import random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("MENTAL HEALTH SUPPORT CHATBOT - SETUP")
print("="*60)
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ============================================
# STEP 2: Create Empathetic Dataset
# ============================================
print("\n📚 Creating Empathetic Dialogues dataset...")

# Empathetic response templates
empathetic_responses = {
    "anxiety": [
        "I hear that you're feeling anxious. That can be really tough. Would you like to talk about what's worrying you? I'm here to listen.",
        "It's completely normal to feel anxious sometimes. Can you tell me more about what's making you feel this way?",
        "Anxiety is challenging, but you're not alone. Let's take a deep breath together. What might help right now?",
    ],
    "stress": [
        "Stress can really take a toll on us. Would you like to brainstorm some stress management techniques together?",
        "I hear that you're feeling stressed. Taking small breaks throughout the day might help.",
        "That's a lot to handle. Remember to be kind to yourself during stressful times.",
    ],
    "loneliness": [
        "Feeling lonely is really difficult. I'm here with you right now, and I hear what you're saying.",
        "It takes courage to admit feeling lonely. You're not alone - I'm here to listen.",
        "Connection is so important. Have you thought about reaching out to someone you trust?",
    ],
    "sadness": [
        "I'm really sorry you're feeling this way. Sadness can be heavy to carry alone. I'm here with you.",
        "Thank you for sharing your feelings. It's okay to not be okay.",
        "I hear the pain in your words. You don't have to go through this alone.",
    ],
    "positive": [
        "That's wonderful to hear! I'm genuinely happy for you. Tell me more about what's going well.",
        "I'm so glad you're feeling this way! Celebrating positive moments is so important.",
        "That's amazing! You should be proud of yourself.",
    ]
}

# Context templates
contexts = {
    "anxiety": [
        "I can't stop worrying about my job interview tomorrow",
        "My anxiety has been keeping me up at night",
        "I feel so nervous about speaking in public",
        "What if everything goes wrong? I'm so worried",
    ],
    "stress": [
        "I have so much to do and not enough time",
        "Work has been incredibly stressful lately",
        "I feel like I'm drowning in responsibilities",
        "Everything is piling up and I can't cope",
    ],
    "loneliness": [
        "I feel so alone even when I'm around people",
        "Since moving to a new city, I haven't made any friends",
        "Nobody really understands what I'm going through",
        "I spend most weekends completely alone",
    ],
    "sadness": [
        "I've been feeling really down lately",
        "Nothing seems to make me happy anymore",
        "I feel like crying all the time",
        "I'm struggling to find joy in anything",
    ],
    "positive": [
        "I finally finished a project I've been working on",
        "Today was actually a really good day",
        "I'm feeling hopeful about the future",
        "I practiced self-care today and it helped",
    ]
}

# Generate conversations
random.seed(42)
conversations = []

for i in range(5000):  # 5000 training examples
    category = random.choice(['anxiety', 'stress', 'loneliness', 'sadness', 'positive'])
    context = random.choice(contexts[category])
    response = random.choice(empathetic_responses[category])

    conversations.append({
        'context': context,
        'utterance': response,
    })

# Create validation set
valid_conversations = []
for i in range(500):
    category = random.choice(['anxiety', 'stress', 'loneliness', 'sadness', 'positive'])
    context = random.choice(contexts[category])
    response = random.choice(empathetic_responses[category])

    valid_conversations.append({
        'context': context,
        'utterance': response,
    })

# Convert to Datasets
train_dataset = Dataset.from_list(conversations)
valid_dataset = Dataset.from_list(valid_conversations)

print(f"✅ Training samples: {len(train_dataset):,}")
print(f"✅ Validation samples: {len(valid_dataset):,}")

print("\n📊 Sample conversations:")
for i in range(2):
    print(f"\nExample {i+1}:")
    print(f"  User: {train_dataset[i]['context']}")
    print(f"  Assistant: {train_dataset[i]['utterance']}")

# ============================================
# STEP 3: Load Model and Tokenizer
# ============================================
print("\n🤖 Loading DistilGPT2 model...")

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ Model loaded on: {device}")
print(f"✅ Model parameters: {model.num_parameters():,}")

# ============================================
# STEP 4: Format and Tokenize Data
# ============================================
print("\n🔄 Formatting conversations...")

def format_conversation(example):
    context = str(example['context']).strip()
    response = str(example['utterance']).strip()
    formatted = f"User: {context}\nAssistant: {response}{tokenizer.eos_token}"
    return {"text": formatted}

# Apply formatting
train_dataset = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
valid_dataset = valid_dataset.map(format_conversation, remove_columns=valid_dataset.column_names)

print("✅ Formatting complete")

# Tokenize
print("\n🔤 Tokenizing datasets...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_valid = valid_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print(f"✅ Training samples: {len(tokenized_train)}")
print(f"✅ Validation samples: {len(tokenized_valid)}")

# ============================================
# STEP 5: Configure Training (CORRECTED PARAMETERS)
# ============================================
print("\n⚙️ Configuring training...")

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training arguments with correct parameter names for latest version
training_args = TrainingArguments(
    output_dir="./empathetic-chatbot",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="steps",  # This works in newer versions
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=3e-5,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True if torch.cuda.is_available() else False,
    report_to="none",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

print("✅ Training ready!")
print(f"  Effective batch size: 16")
print(f"  Total epochs: 3")

# ============================================
# STEP 6: Fine-Tune the Model
# ============================================
print("\n" + "="*60)
print("🚀 STARTING FINE-TUNING")
print("="*60)
print("⏳ This will take 10-15 minutes...\n")

# Train the model
trainer.train()

# Save the model
print("\n💾 Saving fine-tuned model...")
trainer.save_model("./empathetic-chatbot-final")
tokenizer.save_pretrained("./empathetic-chatbot-final")
print("✅ Model saved to './empathetic-chatbot-final'")

# ============================================
# STEP 7: Create Response Generator
# ============================================
print("\n🎯 Creating empathetic response generator...")

def get_response(user_input, temperature=0.7):
    """Generate an empathetic response."""
    # Format prompt
    prompt = f"User: {user_input}\nAssistant:"
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=80,
            temperature=temperature,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("Assistant:")[-1].strip()

    # Clean up
    if "User:" in response:
        response = response.split("User:")[0].strip()

    # Crisis safety filter
    crisis_words = ["suicide", "kill myself", "want to die", "end my life", "self-harm"]
    if any(word in user_input.lower() for word in crisis_words):
        response = "⚠️ I'm really concerned about you. Please reach out to the National Suicide Prevention Lifeline at **988** immediately. You matter, and help is available. 💙"

    # Default response if empty
    if len(response) < 3:
        response = "I'm here for you. Could you tell me more about what you're experiencing?"

    return response

print("✅ Response generator ready!")

# ============================================
# STEP 8: Test the Model
# ============================================
print("\n" + "="*60)
print("🧪 TESTING THE MODEL")
print("="*60)

test_messages = [
    "I'm feeling really anxious about my presentation tomorrow",
    "I've been so lonely since moving to a new city",
    "Work is stressing me out so much lately",
    "I'm proud of myself for getting through today",
    "I feel so overwhelmed with everything I need to do"
]

print("\nTesting responses:\n")
for i, msg in enumerate(test_messages, 1):
    response = get_response(msg)
    print(f"{i}. 👤 User: {msg}")
    print(f"   🤗 Assistant: {response}")
    print()

# ============================================
# STEP 9: Interactive Chat Interface
# ============================================
def chat():
    """Start interactive chat session."""
    print("\n" + "="*60)
    print("🧠 MINDFULMATE - Mental Health Support Chatbot")
    print("="*60)
    print("\n🌟 Hello! I'm here to listen and support you.")
    print("💬 You can talk to me about anxiety, stress, loneliness, or anything on your mind.")
    print("\n📝 Commands:")
    print("   • 'quit' or 'exit' - End the conversation")
    print("   • 'clear' - Reset the conversation")
    print("   • 'help' - Show this message")
    print("\n" + "-"*60)

    while True:
        user_input = input("\n💭 You: ").strip()

        if user_input.lower() in ['quit', 'exit']:
            print("\n🤗 Assistant: Thank you for talking with me. Remember, taking care of your mental health is important. You're not alone. Wishing you peace and strength. 💙")
            break

        elif user_input.lower() == 'clear':
            print("\n🤗 Assistant: Conversation reset. How are you feeling today?")
            continue

        elif user_input.lower() == 'help':
            print("\n📖 Commands: quit, exit, clear, help")
            print("Simply type your thoughts and I'll respond with support and empathy.")
            continue

        elif not user_input:
            print("🤗 Assistant: I'm here whenever you're ready to talk.")
            continue

        # Generate and print response
        print("🤗 Assistant: ", end="", flush=True)
        response = get_response(user_input)
        print(response)

# ============================================
# STEP 10: Demo Function
# ============================================
def run_demo():
    """Run a demo of the chatbot with preset messages."""
    print("\n" + "="*60)
    print("📋 DEMO MODE - Sample Conversations")
    print("="*60)

    demo_messages = [
        "I'm feeling really anxious today",
        "I feel so alone right now",
        "I'm proud of my progress in therapy",
        "Work has been so stressful lately",
        "I'm having a really hard day"
    ]

    for msg in demo_messages:
        print(f"\n👤 User: {msg}")
        print(f"🤗 Assistant: {get_response(msg)}")
        print("-" * 50)

# ============================================
# MAIN EXECUTION
# ============================================
if __name__ == "__main__":
    print("\n" + "="*60)
    print("🎉 MODEL FINE-TUNING COMPLETE!")
    print("="*60)

    print("\n📌 Choose an option:")
    print("   1️⃣ Start interactive chat")
    print("   2️⃣ Run demo mode")
    print("   3️⃣ Test with custom message")

    choice = input("\nEnter your choice (1/2/3): ").strip()

    if choice == "1":
        chat()
    elif choice == "2":
        run_demo()
    elif choice == "3":
        custom_msg = input("\n💭 Your message: ")
        print(f"\n🤗 Assistant: {get_response(custom_msg)}")
    else:
        print("\nStarting interactive chat by default...")
        chat()

MENTAL HEALTH SUPPORT CHATBOT - SETUP
✅ PyTorch version: 2.11.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4

📚 Creating Empathetic Dialogues dataset...
✅ Training samples: 5,000
✅ Validation samples: 500

📊 Sample conversations:

Example 1:
  User: I can't stop worrying about my job interview tomorrow
  Assistant: Anxiety is challenging, but you're not alone. Let's take a deep breath together. What might help right now?

Example 2:
  User: Since moving to a new city, I haven't made any friends
  Assistant: Feeling lonely is really difficult. I'm here with you right now, and I hear what you're saying.

🤖 Loading DistilGPT2 model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded on: cuda
✅ Model parameters: 81,912,576

🔄 Formatting conversations...


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



💾 Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


✅ Model saved to './empathetic-chatbot-final'

🎯 Creating empathetic response generator...
✅ Response generator ready!

🧪 TESTING THE MODEL

Testing responses:

1. 👤 User: I'm feeling really anxious about my presentation tomorrow
   🤗 Assistant: It's completely normal to feel anxious sometimes. Can you tell me more about what's making you feel this way? I'm here to listen. Tell me more if you're feeling anxious. I'm here to listen.

2. 👤 User: I've been so lonely since moving to a new city
   🤗 Assistant: It takes courage to admit feeling lonely. You're not alone - I'm here to listen. Let's take a deep breath together. What might help right now? Tell me more about what's going well. Would you like to talk about what's worrying you? I hear that you're feeling anxious. That can be really tough. Can you tell me more about what's worrying you? I'm

3. 👤 User: Work is stressing me out so much lately
   🤗 Assistant: Stress can really take a toll on us. Would you like to brainstorm some stres